# Лабораторная работа №10

## Уравнения эллиптического типа. Вариант 21

Решается задача `U_xx + U_yy = y(10 - 2x)` на области `[0,10]x[0,10]` с граничными условиями `U=x+y` на границе. Используются методы простой итерации и Зейделя на сетках `5x5` и `10x10` с точностью `0.01`.

## Реализация методов

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A, B = 0.0, 10.0
C, D = 0.0, 10.0
EPS = 0.01


def f(x, y):
    return y * (10.0 - 2.0 * x)


def initialize_grid(cells):
    x = np.linspace(A, B, cells + 1)
    y = np.linspace(C, D, cells + 1)
    h = (B - A) / cells
    u = np.zeros((cells + 1, cells + 1))
    u[:, 0] = x + C
    u[:, -1] = x + D
    u[0, :] = A + y
    u[-1, :] = B + y
    return u, x, y, h


def jacobi(cells, eps=EPS):
    u, x, y, h = initialize_grid(cells)
    error = eps + 1
    iterations = 0
    h2 = h * h

    while error > eps:
        old = u.copy()
        new = old.copy()
        error = 0.0
        for i in range(1, cells):
            for j in range(1, cells):
                value = 0.25 * (
                    old[i + 1, j] + old[i - 1, j] +
                    old[i, j + 1] + old[i, j - 1] -
                    h2 * f(x[i], y[j])
                )
                new[i, j] = value
                error = max(error, abs(value - old[i, j]))
        u = new
        iterations += 1

    return u, x, y, h, iterations, error


def seidel(cells, eps=EPS):
    u, x, y, h = initialize_grid(cells)
    error = eps + 1
    iterations = 0
    h2 = h * h

    while error > eps:
        old = u.copy()
        error = 0.0
        for i in range(1, cells):
            for j in range(1, cells):
                value = 0.25 * (
                    u[i + 1, j] + u[i - 1, j] +
                    u[i, j + 1] + u[i, j - 1] -
                    h2 * f(x[i], y[j])
                )
                u[i, j] = value
                error = max(error, abs(value - old[i, j]))
        iterations += 1

    return u, x, y, h, iterations, error


def center_value(u, x, y):
    target_x = 5.0
    target_y = 5.0
    i1 = int(np.searchsorted(x, target_x))
    j1 = int(np.searchsorted(y, target_y))
    if i1 < len(x) and np.isclose(x[i1], target_x) and j1 < len(y) and np.isclose(y[j1], target_y):
        return u[i1, j1]

    i0 = max(0, i1 - 1)
    j0 = max(0, j1 - 1)
    i1 = min(len(x) - 1, i1)
    j1 = min(len(y) - 1, j1)
    tx = (target_x - x[i0]) / (x[i1] - x[i0])
    ty = (target_y - y[j0]) / (y[j1] - y[j0])
    return (
        (1 - tx) * (1 - ty) * u[i0, j0] +
        tx * (1 - ty) * u[i1, j0] +
        (1 - tx) * ty * u[i0, j1] +
        tx * ty * u[i1, j1]
    )


## Численные результаты

In [ ]:
results = []
for cells_count in (5, 10):
    for name, solver in [('Якоби', jacobi), ('Зейдель', seidel)]:
        u, x, y, h, iterations, error = solver(cells_count)
        results.append((cells_count, name, u, x, y, h, iterations, error))
        print(
            f'{name}, сетка {cells_count}x{cells_count}: '
            f'h={h:.2f}, iterations={iterations}, error={error:.6f}, '
            f'U(5,5)={center_value(u, x, y):.6f}'
        )


Якоби, сетка 5x5: h=2.00, iterations=29, error=0.008955, U(5,5)=9.970640
Зейдель, сетка 5x5: h=2.00, iterations=20, error=0.007863, U(5,5)=9.987812
Якоби, сетка 10x10: h=1.00, iterations=88, error=0.009755, U(5,5)=9.807219
Зейдель, сетка 10x10: h=1.00, iterations=60, error=0.009993, U(5,5)=9.905341


## Таблицы значений

In [ ]:
for cells_count, name, u, x, y, h, iterations, error in results:
    print(f'\n{name}, сетка {cells_count}x{cells_count}')
    print(np.round(u.T[::-1], 2))



Якоби, сетка 5x5
[[ 10.    12.    14.    16.    18.    20.  ]
 [  8.   -66.85 -23.87  49.84  92.83  18.  ]
 [  6.   -71.51 -28.47  50.41  93.47  16.  ]
 [  4.   -52.71 -20.86  38.81  70.67  14.  ]
 [  2.   -26.45  -9.07  23.04  40.42  12.  ]
 [  0.     2.     4.     6.     8.    10.  ]]

Зейдель, сетка 5x5
[[ 10.    12.    14.    16.    18.    20.  ]
 [  8.   -66.84 -23.86  49.85  92.83  18.  ]
 [  6.   -71.5  -28.45  50.43  93.49  16.  ]
 [  4.   -52.7  -20.85  38.82  70.68  14.  ]
 [  2.   -26.45  -9.07  23.05  40.43  12.  ]
 [  0.     2.     4.     6.     8.    10.  ]]

Якоби, сетка 10x10
[[ 10.    11.    12.    13.    14.    15.    16.    17.    18.    19.    20.  ]
 [  9.   -38.88 -46.87 -34.92 -12.71  13.94  40.6   62.82  74.8   66.84  19.  ]
 [  8.   -56.65 -71.67 -57.08 -25.86  12.89  51.64  82.9   97.53  82.58  18.  ]
 [  7.   -60.06 -78.04 -63.88 -30.5   11.84  54.2   87.63 101.86  83.96  17.  ]
 [  6.   -56.53 -74.56 -61.86 -30.08  10.82  51.73  83.57  96.34  78.41  16.  ]


## Графики поверхностей

In [ ]:
for cells_count, name, u, x, y, h, iterations, error in results:
    X, Y = np.meshgrid(x, y, indexing='ij')
    fig = plt.figure(figsize=(7, 4.8))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Y, u, cmap='viridis', edgecolor='none')
    ax.set_title(f'{name}, сетка {cells_count}x{cells_count}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('U')
    plt.show()


## Сечения для сравнения методов

In [ ]:
for cells_count in (5, 10):
    pair = [item for item in results if item[0] == cells_count]
    _, _, u_j, x, y, *_ = pair[0]
    _, _, u_s, *_ = pair[1]
    j = int(np.argmin(np.abs(y - 5.0)))

    plt.figure(figsize=(7, 4.5))
    plt.plot(x, u_j[:, j], 'o-', label='Якоби')
    plt.plot(x, u_s[:, j], 's--', label='Зейдель')
    plt.title(f'Сечение y=5, сетка {cells_count}x{cells_count}')
    plt.xlabel('x')
    plt.ylabel('U(x,5)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


## Вывод

Методы Якоби и Зейделя дают близкие численные решения. Метод Зейделя сходится быстрее, поскольку использует уже обновленные значения на текущей итерации. На сетке `10x10` решение получается более детализированным, чем на сетке `5x5`.